## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

In [51]:
import pandas as pd
import numpy as np
import nltk
import re

In [52]:
#Loading dataset
data=pd.read_csv('data/all_kindle_review.csv',index_col=0)

In [53]:
#Relivent columns
data=data[['reviewText','rating']]

In [54]:
#checking missing values
data.isna().sum()

reviewText    0
rating        0
dtype: int64

In [55]:
#Unique value in ratings
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [56]:
#checking for imbalance dataset
data['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

### Preprocessing and cleaning

In [57]:
#positive review is 1 and negative review is 0
data['rating']=data['rating'].apply(lambda x: 0 if x<3 else 1)

In [58]:
data['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [59]:
#Lowering the review text
data['reviewText']=data['reviewText'].str.lower()

In [60]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [61]:
# special characters
data['reviewText'] = data['reviewText'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9]+', ' ', str(x))
)

# URL
data['reviewText'] = data['reviewText'].apply(
    lambda x: re.sub(
        r'https?://\S+|www\.\S+',
        '',
        str(x)
    )
)

# HTML tags
data['reviewText'] = data['reviewText'].apply(
    lambda x: BeautifulSoup(str(x), 'html.parser').get_text()
)

# additional spaces
data['reviewText'] = data['reviewText'].apply(
    lambda x: " ".join(x.split())
)

In [62]:
data.head()

,reviewText,rating
0,jace rankin may be short but he s nothing to m...,1
1,great short read i didn t want to put it down ...,1
2,i ll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [63]:
#Lemmatizer
from nltk.stem import WordNetLemmatizer

In [64]:
lemma=WordNetLemmatizer()

In [65]:
#lemmatizing the text
def lemma_word(text):
    return " ".join([ lemma.lemmatize(word) for word in text.split()])

In [66]:
data['reviewText'] = data['reviewText'].apply(
    lambda x: lemma_word(x))

In [67]:
data['reviewText'].head()

0    jace rankin may be short but he s nothing to m...
1    great short read i didn t want to put it down ...
2    i ll start by saying this is the first of four...
3    aggie is angela lansbury who carry pocketbook ...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: str

## Train test split

In [68]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data['reviewText'],
    data['rating'],
    test_size=0.2,
    random_state=42,
    stratify=data['rating']
)

In [69]:
train_sentences = X_train.apply(str.split).tolist()

In [70]:
test_sentences=X_test.apply(str.split).tolist()

In [71]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0
)

In [72]:
model.wv['great'].shape

(100,)

In [73]:
def avg_word2vec(doc):
    vectors = [
        model.wv[word]
        for word in doc
        if word in model.wv.key_to_index
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [74]:
from tqdm import tqdm

X_train_avg = []

for i in tqdm(range(len(train_sentences))):
    X_train_avg.append(avg_word2vec(train_sentences[i]))

100%|██████████| 9600/9600 [00:03<00:00, 2546.11it/s]


In [75]:
X_test_avg = []

for i in tqdm(range(len(test_sentences))):
    X_test_avg.append(avg_word2vec(test_sentences[i]))

100%|██████████| 2400/2400 [00:00<00:00, 2672.27it/s]


In [76]:
X_train_avg = np.array(X_train_avg)
X_test_avg = np.array(X_test_avg)

In [77]:
print(X_train_avg.shape)
print(X_test_avg.shape)

(9600, 100)
(2400, 100)


In [78]:
df = pd.DataFrame(X_train_avg)
df['rating'] = y_train.values

In [79]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,rating
0,0.385538,-0.233110,-0.378182,0.219372,0.131585,0.230939,0.535045,0.534108,0.338312,-0.525491,...,0.068212,-0.083604,-0.058753,-0.098975,0.141247,-0.266416,0.378347,-0.301270,0.222373,1
1,0.434526,-0.149547,-0.284066,0.390046,0.029904,0.134649,0.468425,0.748536,0.281984,-0.354495,...,0.152430,-0.070620,0.297279,0.175218,0.106310,0.014127,0.478732,-0.379586,0.418832,0
2,0.742591,-0.566308,-0.161310,0.433516,-0.091745,0.195086,0.602062,0.771292,0.502387,-0.195774,...,0.146197,-0.020808,0.244535,0.209289,0.135651,-0.072553,0.559112,-0.332387,0.415331,0
3,0.595077,-0.322483,-0.454182,0.139386,0.010801,0.068878,0.537660,0.870377,0.482307,-0.311125,...,0.103914,0.216803,0.309828,0.138685,-0.056442,0.162717,0.471457,-0.110523,-0.116524,0
4,0.369700,-0.216776,-0.117977,0.395944,0.013747,0.163159,0.550430,0.592522,0.218875,-0.260032,...,0.271921,-0.126457,0.069200,0.199141,0.232957,-0.276635,0.546705,-0.335417,0.240633,1


In [80]:
from sklearn.ensemble import RandomForestClassifier

random = RandomForestClassifier()

random.fit(X_train_avg, y_train)

y_pred = random.predict(X_test_avg)

In [81]:
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))

0.7645833333333333


In [82]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.71      0.50      0.58       800
           1       0.78      0.90      0.84      1600

    accuracy                           0.76      2400
   macro avg       0.75      0.70      0.71      2400
weighted avg       0.76      0.76      0.75      2400



In [83]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')

svm.fit(X_train_avg, y_train)

y_pred_svm = svm.predict(X_test_avg)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.8008333333333333
              precision    recall  f1-score   support

           0       0.75      0.61      0.67       800
           1       0.82      0.90      0.86      1600

    accuracy                           0.80      2400
   macro avg       0.78      0.75      0.76      2400
weighted avg       0.80      0.80      0.79      2400



In [84]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_avg, y_train)

y_pred_lr = lr.predict(X_test_avg)

In [85]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.7995833333333333
              precision    recall  f1-score   support

           0       0.75      0.59      0.66       800
           1       0.82      0.90      0.86      1600

    accuracy                           0.80      2400
   macro avg       0.78      0.75      0.76      2400
weighted avg       0.80      0.80      0.79      2400

